# 02 — Preparação automática e proteção do TEST

**Modo padrão: preservar a base existente.** O notebook audita a estrutura, registra/confere
os hashes do TEST e gera automaticamente o ZIP validado para o Colab. Não apaga a pasta
`homeobjects_yolo`, não redistribui o TEST e não lê planilhas preenchidas manualmente.

O modo `novo_dataset` prepara uma **nova versão** em outra pasta. Nele, imagens e anotações
idênticas são deduplicadas; coincidências de dHash formam grupos conservadores, mantidos
juntos em train ou val. Grupos relacionados ao TEST ficam fora do desenvolvimento.
dHash igual não é tratado como prova de que duas imagens são o mesmo objeto/cena.

A base original e os resultados históricos permanecem preservados. Reagrupar dados de
desenvolvimento exige novo treinamento; não é necessário para testar o modelo atual no 05d.


In [1]:
import sys
from pathlib import Path
import json
import yaml
from IPython.display import display
from datetime import datetime

atual = Path.cwd().resolve()
RAIZ_PROJETO = next((p for p in [atual, *atual.parents]
                     if (p / 'src/preparacao_dados.py').is_file()), None)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Copie os arquivos src da revisão e abra o notebook dentro do projeto.')
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))
from src.auditoria_dados import CLASSES_HOMEOBJECTS, localizar_raiz_dataset
from src.preparacao_dados import (proteger_dataset_existente, criar_dataset_automatico,
                                 exportar_zip_validado, salvar_json)


## 1. Configuração

Deixe `MODO = "preservar"` para continuar com o baseline atual. Use `novo_dataset` somente
quando quiser preparar uma nova versão de desenvolvimento para um novo treinamento.
Isso é uma configuração de experimento, não preenchimento por imagem.


In [2]:
MODO = 'novo_dataset'  # alternativa: 'novo_dataset'
cfg = yaml.safe_load((RAIZ_PROJETO / 'configs/automacao.yaml').read_text(encoding='utf-8'))
original_path = RAIZ_PROJETO / 'configs/projeto.yaml'
original = yaml.safe_load(original_path.read_text(encoding='utf-8')) if original_path.exists() else {}
CLASSES = {int(k): str(v) for k, v in original.get('classes', CLASSES_HOMEOBJECTS).items()}
BASE_EXISTENTE = RAIZ_PROJETO / cfg['dataset_base']
PASTA_RAW = RAIZ_PROJETO / original.get('dataset', {}).get('pasta_raw', cfg['pasta_raw'])
PASTA_RELATORIOS = RAIZ_PROJETO / 'reports/conferencia_preparacao' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
if MODO not in {'preservar', 'novo_dataset'}:
    raise ValueError('MODO deve ser preservar ou novo_dataset.')
print('Modo:', MODO)
print('Base existente:', BASE_EXISTENTE)


Modo: novo_dataset
Base existente: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\data\processed\homeobjects_yolo


## 2. Auditoria da base existente e referência de preservação

Na primeira execução, o TEST existente é adotado como referência por hashes e classes.
Nas seguintes, qualquer alteração de conteúdo, nomes ou classes bloqueia o fluxo.
A conferência da quantidade de 403 imagens complementa os hashes; sozinha não prova que
as imagens são as mesmas. Erros críticos das anotações também bloqueiam a continuidade.


In [3]:
auditoria_existente, referencia_test = proteger_dataset_existente(
    BASE_EXISTENTE, PASTA_RELATORIOS, CLASSES, cfg['imagens_test'])
display(auditoria_existente['resumo'])
print('TEST conferido:', referencia_test['imagens'], 'imagens')


Auditando train: 1822 imagens
  train: 1/1822
  train: 250/1822
  train: 500/1822
  train: 750/1822
  train: 1000/1822
  train: 1250/1822
  train: 1500/1822
  train: 1750/1822
  train: 1822/1822
Auditando val: 456 imagens
  val: 1/456
  val: 250/456
  val: 456/456
Auditando test: 403 imagens
  test: 1/403
  test: 250/403
  test: 403/403


,split,imagens,imagens_corrompidas,labels_ausentes,labels_vazios
0,train,1822,0,0,0
1,val,456,0,0,0
2,test,403,0,0,0


TEST conferido: 403 imagens


## 3. Preservar ou criar nova versão

No modo preservar não há cópia nem nova divisão. No modo novo_dataset, a auditoria é
refeita nos dados brutos para evitar usar um CSV antigo. A divisão por grupos busca a
proporção 80/20 e exige todas as classes em train e val. As quantidades podem diferir
ligeiramente dos alvos, porque grupos inteiros não podem ser quebrados.

Se uma classe só existe em um grupo, a preparação é bloqueada: não há divisão válida
com essa classe nos dois conjuntos sem quebrar o grupo. Não há seleção manual de imagens.


In [4]:
if MODO == 'preservar':
    DATASET_PARA_USAR = BASE_EXISTENTE
    resumo_preparacao = {'modo': MODO, 'dataset': str(DATASET_PARA_USAR),
                         'test_sha256': referencia_test['sha256']}
else:
    raiz_raw = localizar_raiz_dataset(PASTA_RAW)
    if raiz_raw is None:
        raise FileNotFoundError('Dados brutos não encontrados. Execute primeiro a auditoria 01.')
    DATASET_PARA_USAR, resumo_preparacao = criar_dataset_automatico(
        raiz_raw, BASE_EXISTENTE, RAIZ_PROJETO, CLASSES,
        proporcao=cfg['proporcao_validacao'], semente=cfg['semente'],
        esperado_test=cfg['imagens_test'])
print('Dataset selecionado:', DATASET_PARA_USAR)
print(json.dumps(resumo_preparacao, ensure_ascii=False, indent=2))


Auditando train: 1822 imagens
  train: 1/1822
  train: 250/1822
  train: 500/1822
  train: 750/1822
  train: 1000/1822
  train: 1250/1822
  train: 1500/1822
  train: 1750/1822
  train: 1822/1822
Auditando val: 456 imagens
  val: 1/456
  val: 250/456
  val: 456/456
Auditando test: 403 imagens
  test: 1/403
  test: 250/403
  test: 403/403
Auditando train: 2285 imagens
  train: 1/2285
  train: 250/2285
  train: 500/2285
  train: 750/2285
  train: 1000/2285
  train: 1250/2285
  train: 1500/2285
  train: 1750/2285
  train: 2000/2285
  train: 2250/2285
  train: 2285/2285
Auditando val: 404 imagens
  val: 1/404
  val: 250/404
  val: 404/404
Auditando train: 1824 imagens
  train: 1/1824
  train: 250/1824
  train: 500/1824
  train: 750/1824
  train: 1000/1824
  train: 1250/1824
  train: 1500/1824
  train: 1750/1824
  train: 1824/1824
Auditando val: 456 imagens
  val: 1/456
  val: 250/456
  val: 456/456
Auditando test: 403 imagens
  test: 1/403
  test: 250/403
  test: 403/403
Dataset selecionado

## 4. Gerar automaticamente o pacote para o Colab

O ZIP contém imagens, labels, YAML e um manifesto de verificação. Somente arquivos de
dados necessários são empacotados. O notebook 03 verifica os bytes antes do treinamento.
A geração do ZIP exige espaço livre; não modifica os dados de origem.


In [5]:
ZIP_DATASET = exportar_zip_validado(
    DATASET_PARA_USAR, RAIZ_PROJETO / 'exports/datasets', CLASSES,
    esperado_test=cfg['imagens_test'], test_sha256_esperado=referencia_test['sha256'])
ultima = {'modo': MODO, 'dataset': str(DATASET_PARA_USAR), 'zip': str(ZIP_DATASET),
          'test_sha256': referencia_test['sha256'], 'imagens_test': referencia_test['imagens']}
salvar_json(RAIZ_PROJETO / 'reports/automacao/ultima_preparacao.json', ultima)
print('ZIP validado:', ZIP_DATASET)
print('Para um novo treinamento no Colab, disponibilizar este ZIP no Drive.')


Auditando train: 1824 imagens
  train: 1/1824
  train: 250/1824
  train: 500/1824
  train: 750/1824
  train: 1000/1824
  train: 1250/1824
  train: 1500/1824
  train: 1750/1824
  train: 1824/1824
Auditando val: 456 imagens
  val: 1/456
  val: 250/456
  val: 456/456
Auditando test: 403 imagens
  test: 1/403
  test: 250/403
  test: 403/403
ZIP validado: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\exports\datasets\20261013_001554_110922\homeobjects_yolo_validado.zip
Para um novo treinamento no Colab, disponibilizar este ZIP no Drive.


## Próximo passo

Para usar o modelo já treinado: execute o notebook **05d**. Não é necessário executar o 03.
Para novo treinamento, disponibilizar `homeobjects_yolo_validado.zip` à pasta de datasets do Drive
e abra o notebook 03 revisado. Autorizar o Drive e selecionar GPU continuam sendo ações
da sessão do Colab; nenhuma planilha precisa ser preenchida.
